In [10]:
import os
import pandas as pd

folder = r"C:\Users\amira\Downloads\SUC3Dataset"
file_s1 = os.path.join(folder, "Electron_SUC3_S1_DifferentialProtection_Transformer_RMS.csv")
file_s2 = os.path.join(folder, "Electron_SUC3_S2_MITM_FDI_IEC61850_SV_RMS.csv")

print("S1 exists:", os.path.exists(file_s1), file_s1)
print("S2 exists:", os.path.exists(file_s2), file_s2)

# Peek first rows to confirm columns
peek1 = pd.read_csv(file_s1, nrows=5)
peek2 = pd.read_csv(file_s2, nrows=5)

print("\nS1 columns:", peek1.columns.tolist())
print("S2 columns:", peek2.columns.tolist())
print("\nS1 head:\n", peek1)
print("\nS2 head:\n", peek2)


S1 exists: True C:\Users\amira\Downloads\SUC3Dataset\Electron_SUC3_S1_DifferentialProtection_Transformer_RMS.csv
S2 exists: True C:\Users\amira\Downloads\SUC3Dataset\Electron_SUC3_S2_MITM_FDI_IEC61850_SV_RMS.csv

S1 columns: ['Time', 'RMS_value_phA', 'RMS_value_phB', 'RMS_value_phC', 'RMS_value_phA_L1', 'RMS_value_phB_L1', 'RMS_value_phC_L1']
S2 columns: ['Time', 'RMS_value_phA', 'RMS_value_phB', 'RMS_value_phC', 'RMS_value_phA_L1', 'RMS_value_phB_L1', 'RMS_value_phC_L1']

S1 head:
        Time  RMS_value_phA  RMS_value_phB  RMS_value_phC  RMS_value_phA_L1  \
0  0.000000        0.11991       0.121186       0.120563               0.0   
1  0.000005        0.11991       0.121186       0.120563               0.0   
2  0.000010        0.11991       0.121186       0.120563               0.0   
3  0.000015        0.11991       0.121186       0.120563               0.0   
4  0.000020        0.11991       0.121186       0.120563               0.0   

   RMS_value_phB_L1  RMS_value_phC_L1  
0  

In [ ]:
import pandas as pd
import numpy as np

def scan_file_basic(path, label_name, chunksize=500_000):
    total_rows = 0
    time_min, time_max = None, None

    for chunk in pd.read_csv(path, chunksize=chunksize):
        total_rows += len(chunk)

 
        t_num = pd.to_numeric(chunk["Time"], errors="coerce")
        if t_num.notna().mean() > 0.8:
            mn, mx = float(t_num.min()), float(t_num.max())
        else:

            t_td = pd.to_timedelta(chunk["Time"], errors="coerce")
            if t_td.notna().mean() > 0.2:
                mn, mx = float(t_td.dt.total_seconds().min()), float(t_td.dt.total_seconds().max())
            else:

                mn, mx = str(chunk["Time"].min()), str(chunk["Time"].max())

        if time_min is None:
            time_min, time_max = mn, mx
        else:

            if isinstance(time_min, float) and isinstance(mn, float):
                time_min = min(time_min, mn)
                time_max = max(time_max, mx)
            else:

                time_min = min(str(time_min), str(mn))
                time_max = max(str(time_max), str(mx))

    return {"scenario": label_name, "rows": total_rows, "time_min": time_min, "time_max": time_max}

s1_info = scan_file_basic(file_s1, "S1_baseline")
s2_info = scan_file_basic(file_s2, "S2_attack_scenario")

print(s1_info)
print(s2_info)


{'scenario': 'S1_baseline', 'rows': 8000000, 'time_min': 0.0, 'time_max': 39.999996}
{'scenario': 'S2_attack_scenario', 'rows': 8000000, 'time_min': 0.0, 'time_max': 39.999996}


In [13]:
import pandas as pd

columns = pd.read_csv(file_s1, nrows=1).columns.tolist()

def plain_meaning(col):
    c = col.lower()
    if c == "time":
        return "Time of measurement (very high frequency sampling)"
    if "rms_value_pha" in c and "_l1" not in c:
        return "Phase A electrical strength (main channel)"
    if "rms_value_phb" in c and "_l1" not in c:
        return "Phase B electrical strength (main channel)"
    if "rms_value_phc" in c and "_l1" not in c:
        return "Phase C electrical strength (main channel)"
    if "rms_value_pha_l1" in c:
        return "Phase A electrical strength (L1 channel / second measurement point)"
    if "rms_value_phb_l1" in c:
        return "Phase B electrical strength (L1 channel / second measurement point)"
    if "rms_value_phc_l1" in c:
        return "Phase C electrical strength (L1 channel / second measurement point)"
    return "Other"

feature_table = pd.DataFrame({"column": columns, "plain_english_meaning": [plain_meaning(c) for c in columns]})
print(feature_table)


             column                              plain_english_meaning
0              Time  Time of measurement (very high frequency sampl...
1     RMS_value_phA         Phase A electrical strength (main channel)
2     RMS_value_phB         Phase B electrical strength (main channel)
3     RMS_value_phC         Phase C electrical strength (main channel)
4  RMS_value_phA_L1  Phase A electrical strength (L1 channel / seco...
5  RMS_value_phB_L1  Phase B electrical strength (L1 channel / seco...
6  RMS_value_phC_L1  Phase C electrical strength (L1 channel / seco...


In [ ]:
import pandas as pd
import numpy as np

value_cols = [
    "RMS_value_phA", "RMS_value_phB", "RMS_value_phC",
    "RMS_value_phA_L1", "RMS_value_phB_L1", "RMS_value_phC_L1"
]

def chunked_stats(path, scenario_name, chunksize=500_000):

    n = 0
    mean = None
    M2 = None
    vmin = None
    vmax = None

    for chunk in pd.read_csv(path, usecols=["Time"] + value_cols, chunksize=chunksize):
        x = chunk[value_cols].astype(float)

        # init
        if mean is None:
            mean = x.mean().to_numpy()
            M2 = ((x - x.mean())**2).sum().to_numpy()
            vmin = x.min().to_numpy()
            vmax = x.max().to_numpy()
            n = len(x)
            continue

  
        n2 = len(x)
        mean2 = x.mean().to_numpy()
        M2_2 = ((x - x.mean())**2).sum().to_numpy()

        delta = mean2 - mean
        new_n = n + n2
        mean = mean + delta * (n2 / new_n)
        M2 = M2 + M2_2 + (delta**2) * (n * n2 / new_n)

        vmin = np.minimum(vmin, x.min().to_numpy())
        vmax = np.maximum(vmax, x.max().to_numpy())
        n = new_n

    std = np.sqrt(M2 / (n - 1))
    out = pd.DataFrame({
        "feature": value_cols,
        "mean": mean,
        "std": std,
        "min": vmin,
        "max": vmax
    })
    out.insert(0, "scenario", scenario_name)
    return out

s1_stats = chunked_stats(file_s1, "S1_baseline")
s2_stats = chunked_stats(file_s2, "S2_attack_scenario")

stats_all = pd.concat([s1_stats, s2_stats], ignore_index=True)
print(stats_all)


              scenario           feature         mean          std       min  \
0          S1_baseline     RMS_value_phA   173.080422   160.699219  0.117450   
1          S1_baseline     RMS_value_phB   172.872787   160.720168  0.117346   
2          S1_baseline     RMS_value_phC   172.758649   161.030861  0.113664   
3          S1_baseline  RMS_value_phA_L1  3603.599118  3343.379933  0.000000   
4          S1_baseline  RMS_value_phB_L1  3602.009580  3342.362091  0.000000   
5          S1_baseline  RMS_value_phC_L1  3602.465450  3349.628456  0.000000   
6   S2_attack_scenario     RMS_value_phA   102.034229   150.527202  0.138290   
7   S2_attack_scenario     RMS_value_phB   101.943096   150.429942  0.116017   
8   S2_attack_scenario     RMS_value_phC   101.915465   150.400325  0.115577   
9   S2_attack_scenario  RMS_value_phA_L1  2125.046826  3139.438512  0.000000   
10  S2_attack_scenario  RMS_value_phB_L1  2124.841522  3139.172204  0.000000   
11  S2_attack_scenario  RMS_value_phC_L1

In [ ]:
import pandas as pd
import numpy as np

def chunked_simple_features(path, scenario_name, attack_label, chunksize=500_000):
    rows = 0


    s_sum = 0.0
    s_sq = 0.0
    s_min = float("inf")
    s_max = float("-inf")

    m_sum = 0.0
    m_sq = 0.0
    m_min = float("inf")
    m_max = float("-inf")

    for chunk in pd.read_csv(path, usecols=value_cols, chunksize=chunksize):
        A = chunk["RMS_value_phA"].astype(float)
        B = chunk["RMS_value_phB"].astype(float)
        C = chunk["RMS_value_phC"].astype(float)

        A1 = chunk["RMS_value_phA_L1"].astype(float)
        B1 = chunk["RMS_value_phB_L1"].astype(float)
        C1 = chunk["RMS_value_phC_L1"].astype(float)

        phase_spread = (A - B).abs() + (B - C).abs() + (A - C).abs()
        channel_mismatch = (A - A1).abs() + (B - B1).abs() + (C - C1).abs()

        rows += len(chunk)


        s_sum += float(phase_spread.sum())
        s_sq += float((phase_spread**2).sum())
        s_min = min(s_min, float(phase_spread.min()))
        s_max = max(s_max, float(phase_spread.max()))

        m_sum += float(channel_mismatch.sum())
        m_sq += float((channel_mismatch**2).sum())
        m_min = min(m_min, float(channel_mismatch.min()))
        m_max = max(m_max, float(channel_mismatch.max()))


    s_mean = s_sum / rows
    s_std = np.sqrt(max(s_sq / rows - s_mean**2, 0.0))

    m_mean = m_sum / rows
    m_std = np.sqrt(max(m_sq / rows - m_mean**2, 0.0))

    return pd.DataFrame([{
        "scenario": scenario_name,
        "attack": attack_label,
        "rows": rows,
        "phase_spread_mean": s_mean,
        "phase_spread_std": s_std,
        "phase_spread_min": s_min,
        "phase_spread_max": s_max,
        "channel_mismatch_mean": m_mean,
        "channel_mismatch_std": m_std,
        "channel_mismatch_min": m_min,
        "channel_mismatch_max": m_max,
    }])

s1_simple = chunked_simple_features(file_s1, "S1_baseline", 0)
s2_simple = chunked_simple_features(file_s2, "S2_attack_scenario", 1)

simple_compare = pd.concat([s1_simple, s2_simple], ignore_index=True)
print(simple_compare)


             scenario  attack     rows  phase_spread_mean  phase_spread_std  \
0         S1_baseline       0  8000000           7.101999         43.941128   
1  S2_attack_scenario       1  8000000           0.512118          7.215481   

   phase_spread_min  phase_spread_max  channel_mismatch_mean  \
0          0.000765        590.928832           10289.577938   
1          0.032850        311.835280            6068.820197   

   channel_mismatch_std  channel_mismatch_min  channel_mismatch_max  
0           9537.043676              0.354170           22191.10916  
1           8965.391419              0.379255           19350.56110  


In [16]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(42)

def sample_rows(path, n_sample=200_000, chunksize=500_000):
    samples = []
    seen = 0
    for chunk in pd.read_csv(path, usecols=value_cols, chunksize=chunksize):
        seen += len(chunk)
        # take a small random fraction from each chunk
        frac = min(1.0, n_sample / seen)
        take = chunk.sample(frac=frac, random_state=42)
        samples.append(take)

        if sum(len(s) for s in samples) >= n_sample:
            break

    out = pd.concat(samples, ignore_index=True).head(n_sample)
    return out

print("Sampling from S1...")
X1 = sample_rows(file_s1, n_sample=200_000)
y1 = np.zeros(len(X1), dtype=int)

print("Sampling from S2...")
X2 = sample_rows(file_s2, n_sample=200_000)
y2 = np.ones(len(X2), dtype=int)

X = pd.concat([X1, X2], ignore_index=True)
y = np.concatenate([y1, y2])

rf = RandomForestClassifier(
    n_estimators=250,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top important signals:")
print(importance)


Sampling from S1...
Sampling from S2...
Top important signals:
RMS_value_phA       0.291803
RMS_value_phA_L1    0.269620
RMS_value_phB_L1    0.175199
RMS_value_phB       0.170997
RMS_value_phC       0.053143
RMS_value_phC_L1    0.039238
dtype: float64
